# 02 — Métriques Halstead

**Auteur :** Benoît Moraillon — **Licence :** AGPL-3.0

Les métriques Halstead, proposées par Maurice Halstead en 1977, mesurent la **densité d'information** d'un code source à partir d'un comptage des opérateurs et opérandes.

## Définitions de base

Quatre comptages élémentaires :

| Symbole | Définition |
|---|---|
| `n1` | nombre d'opérateurs **distincts** |
| `n2` | nombre d'opérandes **distincts** |
| `N1` | nombre **total** d'occurrences d'opérateurs |
| `N2` | nombre **total** d'occurrences d'opérandes |

Six grandeurs dérivées :

| Grandeur | Formule | Sens |
|---|---|---|
| `vocabulary` | `n = n1 + n2` | Vocabulaire du programme |
| `length` | `N = N1 + N2` | Longueur du programme |
| `volume` | `V = N · log₂(n)` | Quantité d'information à mémoriser |
| `difficulty` | `D = (n1/2) · (N2/n2)` | Niveau de difficulté de lecture |
| `effort` | `E = D · V` | Effort cognitif estimé |
| `estimated_bugs` | `V / 3000` | Estimation indicative du nombre de bugs |

## Pertinence aujourd'hui

Halstead est **contestée** dans la littérature moderne : très indirecte, dépendante du langage, peu corrélée empiriquement aux défauts. Beaucoup d'outils modernes (radon, SonarSource) la conservent par tradition, certains l'ignorent.

Dans KodoNeko, elle pèse seulement **5 %** dans le PrimaryQualityScore. On la garde pour comparer avec l'écosystème, pas comme indicateur principal.

In [ ]:
# Bootstrap : permet l'import des modules sans installation pip
import sys
from pathlib import Path

root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import setup_paths
setup_paths.setup()

## Premier exemple

Un fichier minimal pour exercer Halstead.

In [ ]:
from kodoneko_metrics import count_halstead

source = '''
def somme(a, b):
    c = a + b
    return c
'''

h = count_halstead(source, language="python")
print(f"n1 = {h.n1}  (opérateurs distincts)")
print(f"n2 = {h.n2}  (opérandes distincts)")
print(f"N1 = {h.N1}  (occurrences d'opérateurs)")
print(f"N2 = {h.N2}  (occurrences d'opérandes)")
print()
print(f"Vocabulaire (n)  : {h.vocabulary}")
print(f"Longueur    (N)  : {h.length}")
print(f"Volume      (V)  : {h.volume:.2f}")
print(f"Difficulté  (D)  : {h.difficulty:.2f}")
print(f"Effort      (E)  : {h.effort:.2f}")

## Halstead distingue ce que NCLOC ne voit pas

Deux fonctions de même NCLOC, mais avec une densité d'opérateurs différente.

In [ ]:
leger = '''
def afficher():
    print("a")
    print("b")
    print("c")
'''

dense = '''
def calculer(a, b, c, d):
    return (a + b) * (c - d) / (a * b + c * d)
'''

h1 = count_halstead(leger, language="python")
h2 = count_halstead(dense, language="python")
print(f"Léger : V = {h1.volume:>7.2f}, D = {h1.difficulty:.2f}")
print(f"Dense : V = {h2.volume:>7.2f}, D = {h2.difficulty:.2f}")

La fonction `calculer` est plus courte que `afficher` en NCLOC, mais elle concentre davantage d'opérateurs et d'opérandes : son volume Halstead est plus élevé. C'est la signature d'un calcul dense.

## Ce que Halstead ne capture pas

Halstead ignore la **structure** du code : un `if` ne pèse pas plus qu'un `+`. Une fonction très imbriquée n'apparaîtra pas spécialement "difficile" du point de vue Halstead. C'est ce que McCabe (notebook 03) et Understandability (notebook 04) viennent corriger.

## API de référence

| Fonction | Retour |
|---|---|
| `count_halstead(source, language=...)` | `HalsteadMetrics` |


---

# Mesurer sur un dépôt réel, et dans le temps

Le volume de Halstead mesure la quantité d'information que porte un programme —
sa densité conceptuelle, indépendamment du nombre de lignes. Appliqué à un
dépôt entier, il révèle où se concentre la matière grise du code. Déroulons-le
sur un projet réel, puis suivons son évolution.

> ⚠️ Cette section clone un dépôt public (nécessite un accès réseau) et, pour
> les langages autres que Python, requiert `tree-sitter`. Sans ces prérequis,
> les cellules affichent un message explicatif et s'interrompent proprement,
> sans erreur.

## 1. Récupérer un dépôt public

Nous clonons un petit projet open-source réel. Le clone est *idempotent* : si le
dépôt est déjà présent localement, on le réutilise sans le retélécharger.

In [ ]:
import subprocess, tempfile
from pathlib import Path

REPO_URL = "https://github.com/gothinkster/django-realworld-example-app.git"
repo_dir = Path(tempfile.gettempdir()) / "kodoneko_demo_repo"

if repo_dir.exists():
    print("Dépôt déjà présent :", repo_dir)
else:
    print("Clonage en cours...")
    res = subprocess.run(
        ["git", "clone", REPO_URL, str(repo_dir)],
        capture_output=True, text=True,
    )
    if res.returncode == 0:
        print("Cloné :", repo_dir)
    else:
        print("Clone impossible (réseau indisponible ?) :")
        print("  ", res.stderr.strip().splitlines()[-1] if res.stderr.strip() else "erreur inconnue")
        repo_dir = None

## 2. Analyser le dépôt complet (instantané)

Premier réflexe : prendre une photographie de l'état actuel. On mesure le volume de Halstead
sur l'ensemble du dépôt, tel qu'il est à l'instant présent (le dernier commit).

In [ ]:
from kodoneko_scanner import scan_repo

if repo_dir:
    report = scan_repo(repo_dir)
    print(f"Fichiers analysés : {report.files_scanned}")
    print(f"  Volume de Halstead total : {report.totals.halstead_volume:,.0f}")
    print(f"  (rappel NCLOC            : {report.totals.ncloc})")
else:
    print("(dépôt indisponible — étape ignorée)")

## 3. Analyser un commit précis

Le moteur temporel `kodoneko_temporal` sait reconstituer l'état du dépôt à
**n'importe quelle référence git** (un tag, une branche, un SHA, ou une
expression comme `HEAD~5`). Ici, on mesure volume de Halstead à ce point de l'histoire, cinq commits
en arrière — un véritable voyage dans le passé du code.

In [ ]:
from kodoneko_temporal import analyze_at_ref

def mesure(path):
    """Applique notre métrique à un état du dépôt."""
    report = scan_repo(path)
    return round(report.totals.halstead_volume, 1)

if repo_dir:
    try:
        point = analyze_at_ref(repo_dir, "HEAD~5", analyzer=mesure)
        print(f"À {point.label} (commit {point.sha[:8]}) : volume de Halstead = {point.result}")
    except Exception as e:
        print(f"Commit indisponible (historique trop court ?) : {e}")
else:
    print("(dépôt indisponible — étape ignorée)")

## 4. Mesurer un écart entre deux moments (*diff temporel*)

La question la plus parlante n'est pas « combien ? » mais « **combien de plus
qu'avant ?** ». En comparant la mesure à deux références, on quantifie ce qu'un
intervalle de développement a réellement produit comme volume de Halstead.

In [ ]:
if repo_dir:
    try:
        avant = analyze_at_ref(repo_dir, "HEAD~10", analyzer=mesure)
        apres = analyze_at_ref(repo_dir, "HEAD", analyzer=mesure)
        delta = apres.result - avant.result
        signe = "+" if delta >= 0 else ""
        # Variation relative en pourcentage (2 décimales), protégée contre la division par zéro
        if avant.result:
            pct = delta / avant.result * 100
            pct_txt = f"{signe}{pct:.2f} %"
        else:
            pct_txt = "n/a (valeur initiale nulle)"
        print(f"volume de Halstead il y a 10 commits : {avant.result}")
        print(f"volume de Halstead aujourd'hui       : {apres.result}")
        print(f"Variation absolue    : {signe}{delta}")
        print(f"Variation relative   : {pct_txt}")
    except Exception as e:
        print(f"Diff indisponible : {e}")
else:
    print("(dépôt indisponible — étape ignorée)")

### L'information s'accumule-t-elle, ou se dilue-t-elle ?

Le volume total grimpe à mesure qu'on ajoute des fonctionnalités. Mais une
hausse brutale peut signaler une complexité qui s'emballe. En traçant le volume
de Halstead mois après mois, on distingue la croissance maîtrisée de
l'accumulation incontrôlée.

In [ ]:
from kodoneko_temporal import analyze_over_windows

if repo_dir:
    serie = analyze_over_windows(repo_dir, analyzer=mesure, strategy="monthly")
    print(f"volume de Halstead à la fin de chaque mois ({len(serie)} fenêtres) :\n")
    maxv = max((p.result for p in serie), default=1) or 1
    precedent = None
    for point in serie:
        barre = "█" * min(40, int(point.result / maxv * 40))
        # Variation par rapport au mois précédent, en % (2 décimales)
        if precedent is None:
            var_txt = "     —  "
        elif precedent:
            pct = (point.result - precedent) / precedent * 100
            var_txt = f"{pct:+6.2f} %"
        else:
            var_txt = "    n/a "
        print(f"  {point.label}  {point.result:>8}  {var_txt}  {barre}")
        precedent = point.result
else:
    print("(dépôt indisponible — étape ignorée)")

## En résumé

Nous venons de parcourir les quatre façons d'interroger le volume de Halstead :

1. **L'instantané** — l'état présent du dépôt entier.
2. **Le commit précis** — la mesure à un point quelconque de l'histoire.
3. **Le diff temporel** — l'écart entre deux moments, qui isole ce qu'une
   période a produit.
4. **La série temporelle** — la trajectoire complète, qui révèle les tendances.

Le même analyseur (`mesure`) a servi pour les quatre : c'est toute la force du
moteur temporel, qui découple *ce qu'on mesure* de *quand on le mesure*.